# Warm-start RQAOA: an ensemble of starts

The notebook shows:
- the `warm_start_ensemble_rqaoa_cpu_v1` card;
- global-flip dedup and multiplicities;
- two real warm-state children on the current graph;
- an equal mixture of physical correlators;
- a negative edge, offset and lift;
- the difference between a mixture and a coherent sum;
- quotas of the shared budget;
- a real GW batch;
- a small series of 12 trials.

The first scenario does not require CVXPY. The GW section runs in the classical
profile.

In [1]:
from pathlib import Path
import copy, json, math, os, tempfile
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT=Path.cwd()
if not (ROOT/"configs").exists(): ROOT=ROOT.parent
os.chdir(ROOT)

from QA_max_cut_lib import WeightedGraph, tools
from QA_max_cut_lib.core.budget import ShotBudget
from QA_max_cut_lib.laboratory import WarmStartRQAOACPU, get_variant
from QA_max_cut_lib.solvers.warm_start_ensemble import WarmStartEnsembleCorrelationSource

print(get_variant("warm_start_ensemble_rqaoa_cpu_v1")["status"])
assert get_variant("warm_start_ensemble_rqaoa_cpu_v1")["runnable"]


{'source_review': 'partially_reviewed', 'scientific_specification': 'specified', 'implementation': 'implemented', 'cpu_validation': 'passed', 'gpu_validation': 'not_supported', 'resume_validation': 'not_supported'}


## 1. Control A: two starts

A=(+,+,−,+), B=(+,−,+,+) in the order `(10,30,80,101)`. A and −A give one key
after global-flip dedup; the multiplicity of A is 3, of B is 1.

In [2]:
graph=WeightedGraph([(10,30,2),(10,80,3),(30,80,5)],nodes=[10,30,80,101])
A={10:1,30:1,80:-1,101:1}; B={10:1,30:-1,80:1,101:1}
record={"graph":graph.snapshot(),"assignments":[
 {"candidate_id":"A","spins":A},
 {"candidate_id":"-A","spins":{k:-v for k,v in A.items()}},
 {"candidate_id":"B","spins":B},
 {"candidate_id":"A2","spins":A},
]}
cfg={
 "schema_version":1,"type":"laboratory.warm_start_rqaoa_cpu","seed":20261102,
 "parameters":{"finish_at":3,"max_reductions":1},
 "correlation_source":{
  "type":"warm_start_ensemble_cpu",
  "parameters":{"layers":1,"max_vertices":18,"dtype":"complex128"},
  "candidate_source":{"type":"provided_by_graph","parameters":{"records":[record]}},
  "starts":{"max_starts":2,"deduplication":"global_flip","on_shortage":"use_available","ranking":"current_cut_then_canonical_spins"},
  "warm_start":{"epsilon":0.25,"mixer":{"type":"rounded_modified"}},
  "objective":{"type":"mean_cut"},
  "estimation":{"mode":"exact","shots_per_evaluation":32},
  "optimizer":{"type":"adam","parameters":{"max_steps":0,"learning_rate":0.05,"gradient_tolerance":0.0},"gradient":{"type":"central_difference","parameters":{"step":0.01}}},
  "initialization":{"type":"fixed_angles","parameters":{"values":[0.0,math.pi/2]}},
  "step_initialization":"restart",
  "correlations":{"mode":"exact","aggregation":"uniform_components"},
 },
 "selection":{"type":"max_abs_correlation","parameters":{"pairs":"edges","zero_sign":1}},
 "finisher":{"type":"exact_cpu","parameters":{"max_vertices":16}},
 "limits":{"shots_total":0},
 "collect":{"reduction_history":True,"parameter_trajectory":False,"local_optimizer_history":False,"correlation_tables":True,"final_reduction_state":True},
 "output":{"report":0,"logs":0},
}
solver=WarmStartRQAOACPU(cfg)
solution=solver.solve(graph)
history=solver.warm_start_history[0]
print("solution",solution)
print("C",solver.cut_value,"b",solver.cut_offset,"E_Ising",solver.ising_constant)
print("unique",history["source"]["unique"],"multiplicities",[c["multiplicity"] for c in history["children"]])
assert solver.cut_value==8.0 and solver.cut_offset==8.0 and solver.ising_constant==-7.5
assert [c["multiplicity"] for c in history["children"]]==[3,1]


solution {10: 1, 30: 1, 101: 1, 80: -1}
C 8.0 b 8.0 E_Ising -7.5
unique 2 multiplicities [3, 1]


## 2. Mixture vs coherent sum

For two qubits an equal mixture of `ψ1,ψ2` gives a uniform P and c=0. The
normalized sum of amplitudes gives a different answer; this catches erroneous
coherent aggregation.

In [3]:
psi1=np.array([1,1,1,1],dtype=complex)/2
psi2=np.array([1,-1,-1,1],dtype=complex)/2
p1=np.abs(psi1)**2; p2=np.abs(psi2)**2
p_mix=(p1+p2)/2
coherent=psi1+psi2; coherent=coherent/np.linalg.norm(coherent)
p_coh=np.abs(coherent)**2
print("P mixture",p_mix,"P coherent",p_coh)
assert np.allclose(p_mix,0.25)
assert not np.allclose(p_coh,p_mix)


P mixture [0.25 0.25 0.25 0.25] P coherent [0.5 0.  0.  0.5]


## 3. Quotas of the shared budget

p=1, M_train=32, M_corr=16, max_steps=2, B=368: for K=3 this gives [1,1,0] and a
reserve of 48; for K=2 it gives [1,1].

In [4]:
qcfg=copy.deepcopy(cfg["correlation_source"])
qcfg["estimation"]={"mode":"shots","shots_per_evaluation":32}
qcfg["optimizer"]["parameters"]["max_steps"]=2
qcfg["correlations"]={"mode":"shots","shots":16,"aggregation":"uniform_components"}
qsrc=WarmStartEnsembleCorrelationSource(qcfg,seed=1)
q3,r3=qsrc._quotas(ShotBudget(368),3,qsrc.config)
q2,r2=qsrc._quotas(ShotBudget(368),2,qsrc.config)
print(q3,r3,q2,r2)
assert q3==[1,1,0] and r3==48 and q2==[1,1] and r2==32


[1, 1, 0] 48 [1, 1] 32


## 4. A real GW batch

One SDP, L roundings, no K separate SDPs. This section requires CVXPY/SCS.

In [5]:
try:
    gwcfg=copy.deepcopy(cfg)
    gwcfg["correlation_source"]["candidate_source"]={"type":"gw_rounds_cpu","parameters":{"num_hyperplanes":8,"solver":"SCS","allow_inaccurate":False}}
    gwcfg["output"]={"report":0,"logs":0}
    gw=WarmStartRQAOACPU(gwcfg); gw.solve(graph)
    source=gw.warm_start_history[0]["source"]
    print("SDP calls",source["source_cost"]["sdp_calls"],"rounds",source["rounds_completed"])
    assert source["source_cost"]["sdp_calls"]==1
except ImportError as exc:
    print("GW section skipped:",exc)


SDP calls 1 rounds 8


## 5. A series of 12 trials

Two graphs × two algorithm seeds × three recipes: ordinary RQAOA, WS with K=1,
WS with K=2. That is 12 trials, not 12 quantum children.

In [6]:
spec=tools.load_json("experiments/benchmark_warm_start_rqaoa.json")
spec.pop("save",None)
series=tools.ExperimentRunner(spec).run()
print(series.summary)
print(json.dumps(series.analysis["by_solver"],indent=2)[:2000])
assert series.analysis["planned"]==12 and series.analysis["valid"]==12


planned=12 valid=12 failed=0 skipped=0 success(valid)=0.3333333333333333 wall=0.347s
{
  "rqaoa_cpu": {
    "planned": 4,
    "valid": 4,
    "failed": 0,
    "skipped": 0,
    "cut": {
      "count": 4,
      "mean": -2.0,
      "median": -2.0,
      "min": -3.0,
      "max": -1.0,
      "stdev": 1.1547005383792515
    },
    "gap": {
      "count": 4,
      "mean": 4.0,
      "median": 4.0,
      "min": 3.0,
      "max": 5.0,
      "stdev": 1.1547005383792515
    },
    "success": 0.0
  },
  "ws_rqaoa_k1": {
    "planned": 4,
    "valid": 4,
    "failed": 0,
    "skipped": 0,
    "cut": {
      "count": 4,
      "mean": 0.75,
      "median": 2.0,
      "min": -3.0,
      "max": 2.0,
      "stdev": 2.5
    },
    "gap": {
      "count": 4,
      "mean": 1.25,
      "median": 0.0,
      "min": 0.0,
      "max": 5.0,
      "stdev": 2.5
    },
    "success": 0.75
  },
  "ws_rqaoa_k2": {
    "planned": 4,
    "valid": 4,
    "failed": 0,
    "skipped": 0,
    "cut": {
      "count": 4,
  

## 6. Comparison and plot

The plot uses only the saved records of the series; the solver/source is not run
again. Differences in quality are an observation, not a promise of advantage.

In [7]:
names=sorted(series.analysis["by_solver"])
means=[series.analysis["by_solver"][n]["cut"]["mean"] for n in names]
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(names,means,color=["#64748b","#2563eb","#7c3aed"])
ax.set_ylabel("mean C_original")
ax.set_title("Small WS-RQAOA series: solver-only quality")
ax.grid(axis="y",alpha=.25)
fig.tight_layout()
out=Path("docs/notebook_previews/13_warm_start_rqaoa.png")
out.parent.mkdir(parents=True,exist_ok=True)
fig.savefig(out,dpi=140,facecolor="white")
plt.close(fig)
print("saved",out)


saved docs\notebook_previews\13_warm_start_rqaoa.png


## Summary

- A: U=2, multiplicities 3/1, C=8, b=8, E_Ising=-7.5.
- The mixture of correlations is statistical, not a coherent sum.
- Quotas depend on K and not on the worker count.
- Real GW: one SDP and L constrained roundings.
- Series: 12 trials, 0 failed (in this environment).
- Checkpoint/resume for multi-start RQAOA is not claimed in a12.